[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/extensor_representations/extensor_representations.ipynb)

# Extensor Representations

This example computes minimal faithful real representations of nondegenerate Clifford algebras as extensors. Left multiplication on an algebraic ideal models each multivector as a linear map on a smaller real state space so that distinct multivectors yield distinct maps and multiplying two multivectors composes their maps. Commuting elements that square to plus one generate a projector onto that ideal, giving real state spaces as small as the smallest faithful real matrix representations, including the real Majorana form of the Dirac gamma matrices.

One extensor `state_readout(Full * state_embedding)` embeds a state into the projected left ideal, multiplies on the left by `Full`, and casts back to the state coordinates. Evaluating that extensor on a multivector yields its action on states, while tracing against `representations(Space, state_map).trace(0, 2)` and solving with `Space.scalar_product(Space)` orthogonally projects any state map back onto `Space` in exact rational arithmetic, as used for spacetime spinors in the [Dirac equation example](../../relativity/dirac/dirac.ipynb).

In matrix notation, a faithful real representation assigns $d \times d$ generator matrices $\gamma_i \in \mathbb{R}^{d \times d}$ satisfying $\gamma_i \gamma_j + \gamma_j \gamma_i = 2\eta_{ij} I_d$, extends them to $2^n$ ordered blade products $\gamma_A$ so $\rho(\sum_A a_A e_A) = \sum_A a_A \gamma_A$ satisfies $\rho(ab) = \rho(a)\rho(b)$, and recovers blade coordinates from a matrix $M$ through the trace sum $a_A = \tfrac{1}{d}\operatorname{tr}(\gamma_A^{-1} M)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
import numpy as np

from numga import Algebra, Extensor, GAType, NumpyContext, Orientation
from numga.algebras import STA, VGA2D
from examples.math.extensor_representations import scenarios

# Two Euclidean directions squaring to plus one, with exact rational arithmetic.
ga = VGA2D
mv = ga.exact.multivector
Full = ga.gatype.full()
State = Full.derive.within(Algebra("y+"))
n_state_components = len(State.output_subspace)

## A smaller space of states

Right multiplication by `(1 + x) / 2` projects onto multivectors that right multiplication by `x` leaves unchanged, and left multiplication by any multivector preserves that condition. A subspace closed under left multiplication by the whole algebra is called a left ideal. In two Euclidean dimensions, the `1` and `y` coordinates determine every element of that ideal because right multiplication by `(1 + x) / 2` supplies the matching `x` and `xy` coordinates, while casting back and doubling restores the halved `1` and `y` weights.

In [ ]:
multivector = mv.x + 2 * mv.y + mv.xy                                    # [] Full+

projector = (1 + mv.x) / 2                                               # [] Full+
# Right multiplication by the projector fills in the linked x and xy coordinates.
state_embedding = State * projector                                      # [] Full+ <- State+
# Casting back and doubling restores the 1 and y weights halved by the projector.
state_readout = (2 * Full).cast(State)                                   # [] State+ <- Full+
ideal = Full * projector                                                 # [] Full+ <- Full+
representations = state_readout(Full * state_embedding)                  # [] State+ <- (Full+, State+)
representation = representations(multivector)                            # [] State+ <- State+

print("multivector: Full")
print(multivector.coefficient_table())
print("extensor representation: State <- State")
print(representation.coefficient_table())

## A representation for every blade

Evaluating `representations` on the four basis blades produces four linearly independent maps on `State`, and linearity extends them to every multivector. Composing two state maps matches the geometric product of the multivectors they represent, whereas grade-dependent operations such as the outer product and reverse depend on the blade basis prior to projection.

In matrix notation, the four basis blades act on $\mathbb{R}^2$ through $I_2$, $\sigma_z$, $\sigma_x$, and $-\sigma_x\sigma_z$, with matrix multiplication $\rho(ab) = \rho(a)\rho(b)$.

In [ ]:
# Output blades label the rows and input blades label the columns of each unary state map.
for blade in Full.output_subspace.masks:
    print(f"{ga.blade_name(blade)}:\n{representations(mv.blade(blade)).coefficient_table()}\n")

In [ ]:
state = 1 + 2 * mv.y                                                     # [] State+
other_multivector = 3 + mv.x - mv.y                                      # [] Full+

other_representation = representations(other_multivector)                # [] State+ <- State+
product_representation = representations(multivector * other_multivector)  # [] State+ <- State+

print("multivector:")
print(multivector.coefficient_table())
print("extensor representation:")
print(representation.coefficient_table())
print("representation applied to the state:")
print(representation(state).coefficient_table())
print("product:")
print((multivector * other_multivector).coefficient_table())
print("the two representations applied in turn:")
print(representation(other_representation(state)).coefficient_table())
print("product acting once:")
print(product_representation(state).coefficient_table())

## Recovering the multivector

Because distinct basis blades have trace-orthogonal actions on `State`, composing a state map with `representations` and tracing over the state isolates the scalar product with the represented multivector. Solving against `Space.scalar_product(Space)` therefore recovers every blade coefficient in `Space`, making `readout` both the trace-pairing adjoint and the left inverse of `representations`.

In matrix notation, with $\gamma_A \in \mathbb{R}^{d \times d}$ the matrix of blade $A$, trace orthogonality $\tfrac{1}{d}\operatorname{tr}(\gamma_A^{-1}\gamma_B) = \delta_{AB}$ extracts each coordinate from a matrix $M$ as $a_A = \tfrac{1}{d}\operatorname{tr}(\gamma_A^{-1} M)$.

In [ ]:
def readout(representations: Extensor, state_map: Extensor, Space: GAType) -> Extensor:
    """Trace pairing against `representations` orthogonally projects `state_map` onto `Space` (`[] Space+`)."""
    pairing = Space.scalar_product(Space)                                # [] Scalar+ <- (Space+, Space+)
    n_state_components = len(representations.output_subspace)
    # Composing with the representations and tracing over the state isolates the scalar product with the sought element.
    return pairing.solve(representations(Space, state_map).trace(0, 2) / n_state_components)  # [] Space+


recovered_multivector = readout(representations, representation, Full)   # [] Full+
pairing = Full.scalar_product(Full)                                      # [] Scalar+ <- (Full+, Full+)
# Tracing out the state across two composed representations yields the normalized scalar product.
trace_pairing = representations(Full, representations).trace(0, 3)       # [] Scalar+ <- (Full+, Full+)
round_trip = pairing.solve(trace_pairing / n_state_components)           # [] Full+ <- Full+

print("multivector:")
print(multivector.coefficient_table())
print("extensor representation:")
print(representation.coefficient_table())
print("recovered multivector:")
print(recovered_multivector.coefficient_table())

## Projecting an arbitrary map

A faithful representation need not span every linear map on its state space. In three Euclidean dimensions, the eight multivector blades act on four real state components, leaving an eight-dimensional complement inside the sixteen-dimensional space of state maps. Applying `representations` after `readout` orthogonally projects any state map onto the represented algebra under the trace pairing, leaving a remainder whose trace with every blade representation vanishes.

In [ ]:
space_ga = Algebra("x+y+z+")
space_mv = space_ga.exact.multivector
SpaceFull = space_ga.gatype.full()
SpaceState = space_ga.gatype.even()

space_projector = (1 + space_mv.z) / 2                                   # [] SpaceFull+
space_embedding = SpaceState * space_projector                           # [] SpaceFull+ <- SpaceState+
space_readout = (2 * SpaceFull).cast(SpaceState)                         # [] SpaceState+ <- SpaceFull+
space_representations = space_readout(SpaceFull * space_embedding)       # [] SpaceState+ <- (SpaceFull+, SpaceState+)

# Grade-zero selection has rank one, whereas every nonzero multivector acts on SpaceState with even rank.
arbitrary_map = SpaceState.select[0].cast(SpaceState)                    # [] SpaceState+ <- SpaceState+
projected_multivector = readout(space_representations, arbitrary_map, SpaceFull)  # [] SpaceFull+
projected_representation = space_representations(projected_multivector)  # [] SpaceState+ <- SpaceState+
remainder = arbitrary_map - projected_representation                     # [] SpaceState+ <- SpaceState+

print("SpaceState:", SpaceState)
print("supplied map:", arbitrary_map.gatype)
print(arbitrary_map.coefficient_table())
print("projected multivector:", projected_multivector.gatype)
print(projected_multivector.coefficient_table())
print("projected representation:", projected_representation.gatype)
print(projected_representation.coefficient_table())
print("remainder:", remainder.gatype)
print(remainder.coefficient_table())

## The state space of each signature

Pairwise commuting blades that square to plus one generate commuting projectors, each halving the state dimension. Their products partition the basis blades into cosets of equal size, so choosing one representative blade per coset gives independent real coordinates while rescaling by the coset size restores the projected weights. In odd dimensions where the central volume element squares to plus one, leaving the central sign unconstrained retains both simple blocks of the split algebra; the [Clifford Algebra Explorer](https://coreality.cc/clifford/) tabulates the corresponding projector classifications.

In matrix notation, each Clifford algebra is isomorphic to a matrix algebra over $\mathbb{R}$, $\mathbb{C}$, or $\mathbb{H}$ (or a direct sum of two copies in split odd dimensions), where complex and quaternionic entries multiply the real state dimension by two and four.

In [ ]:
def state_layout(algebra: Algebra) -> tuple[GAType, tuple[int, ...]]:
    """Commuting unit-square blades partition `algebra` into cosets whose representatives span the minimal real state space `State+`."""
    involutions = ()
    coset_group = {0}
    is_split = algebra.dimension % 2 == 1 and algebra.pseudoscalar_squared == 1
    for blade in algebra.blade_masks:
        if blade in coset_group or algebra.geometric_product(blade, blade).coefficient != 1:
            continue
        if any(
            algebra.geometric_product(blade, other).coefficient
            != algebra.geometric_product(other, blade).coefficient
            for other in involutions
        ):
            continue
        # Leaving the central volume unconstrained retains both simple blocks of a split odd-dimensional algebra.
        if is_split and (blade ^ algebra.pseudoscalar_mask) in coset_group:
            continue
        involutions += (blade,)
        coset_group |= {blade ^ other for other in coset_group}

    # Right multiplication by the projector fills each coset, so one blade per coset gives independent state coordinates.
    representatives = {min(blade ^ other for other in coset_group) for blade in algebra.blade_masks}
    State = algebra.gatype(algebra.subspace.from_masks(representatives), orientation=Orientation.extrinsic)
    return State, involutions


def state_representations(algebra: Algebra) -> tuple[GAType, Extensor]:
    """Left multiplication on the minimal projected left ideal represents `algebra` on `State+` (`[] State+ <- (Full+, State+)`)."""
    mv = algebra.exact.multivector
    Full = algebra.gatype.full()
    State, involutions = state_layout(algebra)
    n_coset_blades = 2 ** len(involutions)

    # Commuting unit-square blades yield commuting projectors whose product selects their common positive ideal.
    projector = mv.scalar()                                              # [] Scalar+
    for blade in involutions:
        projector = projector * (1 + mv.blade(blade)) / 2

    # Left multiplication preserves the right ideal of the projector, and rescaling restores the projected weight.
    state_embedding = State * projector                                  # [] Full+ <- State+
    state_readout = (n_coset_blades * Full).cast(State)                  # [] State+ <- Full+
    representations = state_readout(Full * state_embedding)              # [] State+ <- (Full+, State+)
    return State, representations


print(f"{'signature':12s} {'algebra components':>18s} {'real state components':>23s}")
for n_positive, n_negative in scenarios.signatures:
    signature_ga = Algebra((n_positive, n_negative, 0))
    SignatureState, signature_representations = state_representations(signature_ga)  # [] SignatureState+ <- (Full+, SignatureState+)
    print(f"{str((n_positive, n_negative)):12s} {signature_ga.blade_count:18d} {len(SignatureState.output_subspace):23d}")

## Four-component Euclidean and spacetime actions

Both the three-dimensional Euclidean algebra of signature `(3, 0)` and the spacetime algebra of signature `(3, 1)` act faithfully on four real state components, with output blades labelling the rows and input blades labelling the columns of each generator table below. Composing two spacetime generator tables matches the geometric product of their basis vectors, so each squares to the vector norm and distinct generators anticommute, whereas the opposite spacetime signature `(1, 3)` requires eight real state components in the signature table above.

In the matrix notation of the Dirac equation, the four `(3, 1)` generator tables form a real Majorana representation $\gamma^\mu \in \mathbb{R}^{4 \times 4}$ satisfying $\gamma^\mu\gamma^\nu + \gamma^\nu\gamma^\mu = 2\eta^{\mu\nu} I_4$, whereas signature `(1, 3)` requires complex matrices $\gamma^\mu \in \mathbb{C}^{4 \times 4}$ acting on eight real components.

In [ ]:
for signature in scenarios.representative_signatures:
    representative_ga = Algebra(signature)
    representative_mv = representative_ga.exact.multivector
    RepresentativeState, representative_representations = state_representations(representative_ga)  # [] RepresentativeState+ <- (Full+, RepresentativeState+)
    print(f"Cl{representative_ga.pqr[:2]}")
    for blade in representative_ga.basis_vector_masks:
        print(f"\n{representative_ga.blade_name(blade)}:\n{representative_representations(representative_mv.blade(blade)).coefficient_table()}")
    print()

## Spinors of the spacetime algebra

In the spacetime algebra of signature `(3, 1)` used in the [Dirac equation example](../../relativity/dirac/dirac.ipynb), a spinor is modelled as an eight-component even multivector, and a spacetime vector acts on it by left multiplication paired with right multiplication by `t` so the result stays even. Right multiplication by the spin plane `yx` squares to minus one and commutes with every such generator action, grouping the eight even blades into four pairs in which each blade is followed by its image under `yx`.

In the matrix notation of the Dirac equation, replacing each complex entry $a + ib$ of $\gamma^0 = \operatorname{diag}(1, 1, -1, -1)$ and $\gamma^k = \begin{pmatrix} 0 & \sigma_k \\ -\sigma_k & 0 \end{pmatrix}$ by the real $2 \times 2$ block $\begin{pmatrix} a & -b \\ b & a \end{pmatrix}$ produces the same $8 \times 8$ generator tables.

In [ ]:
sta = STA
sta_mv = sta.exact.multivector
Vector = sta.gatype.vector()
spin_plane = sta_mv.yx                                                   # [] Bivector+
# Even multivectors ordered so each blade is followed immediately by its image under yx.
Spinor = sta.gatype(sta.subspace("1 yx zx zy tz tzyx tx ty"), orientation=Orientation.extrinsic)
quarter_turn = (Spinor * spin_plane).cast(Spinor)                        # [] Spinor+ <- Spinor+
spinor_actions = (Vector * Spinor * sta_mv.t).cast(Spinor)               # [] Spinor+ <- (Vector+, Spinor+)

print("spin plane, from the right:")
print(quarter_turn.coefficient_table())
for blade in sta.basis_vector_masks:
    print(f"\n{sta.blade_name(blade)}:\n{spinor_actions(sta_mv.blade(blade)).coefficient_table()}")

## Complex spinor coordinates

Allowing complex scalar coefficients turns `(1 - 1j * yx) / 2` into an idempotent projector onto the subspace of even multivectors where right multiplication by `yx` matches multiplication by `1j`. The four blades `1`, `zx`, `tz`, and `tx` then furnish independent complex coordinates, casting back and doubling restores their halved weights, and the four spacetime generators act on four complex components.

In the matrix notation of the Dirac equation, the resulting $4 \times 4$ complex tables match the standard Dirac gamma matrices $\gamma^0 = \operatorname{diag}(1, 1, -1, -1)$ and $\gamma^k = \begin{pmatrix} 0 & \sigma_k \\ -\sigma_k & 0 \end{pmatrix}$ entry by entry.

In [ ]:
complex_mv = NumpyContext(sta, dtype=complex).multivector
Even = sta.gatype.even()
DiracSpinor = sta.gatype(sta.subspace("1 zx tz tx"), orientation=Orientation.extrinsic)
complex_projector = (1 - 1j * complex_mv.yx) / 2                         # [] Even+
spinor_embedding = DiracSpinor * complex_projector                       # [] Even+ <- DiracSpinor+
spinor_readout = (2 * Even).cast(DiracSpinor)                            # [] DiracSpinor+ <- Even+
complex_quarter_turn = spinor_readout(spinor_embedding * complex_mv.yx)  # [] DiracSpinor+ <- DiracSpinor+
dirac_actions = spinor_readout(Vector * spinor_embedding * complex_mv.t)  # [] DiracSpinor+ <- (Vector+, DiracSpinor+)

print("spin plane, from the right:")
print(complex_quarter_turn.coefficient_table())
for blade in sta.basis_vector_masks:
    print(f"\n{sta.blade_name(blade)}:\n{dirac_actions(complex_mv.blade(blade)).coefficient_table()}")

In [ ]:
# checks
# Readout and embedding invert each other on states and on the projected ideal, composition matches
# multiplication, trace readout recovers the identity on the algebra, and projection remainders vanish under trace.
for exact_residual in (
    state_readout(state_embedding) - State,
    state_embedding(state_readout(ideal)) - ideal,
    representation(other_representation) - product_representation,
    recovered_multivector - multivector,
    round_trip - Full,
    space_representations(SpaceFull, remainder).trace(0, 2),
    quarter_turn(quarter_turn) + Spinor,
    spinor_actions(Vector, quarter_turn) - quarter_turn(spinor_actions),
):
    assert not exact_residual.kernel.output_nonzero_indices()

# Complex readout inverts embedding and represents right multiplication by yx as multiplication by 1j.
assert np.allclose((spinor_readout(spinor_embedding) - DiracSpinor).kernel, 0, atol=1e-12)
assert np.allclose((complex_quarter_turn - 1j * spinor_readout(spinor_embedding)).kernel, 0, atol=1e-12)